# ON(CT) Finance – Economie notebook (TwelveData + backward engineering)

Doel: met **echte beursdata** ontdekken hoe stockdata eruitziet (via een API-call) en hoe je daaruit informatie haalt om een **(fictief) koop/verkoop-plan** te maken.

> **Let op:** dit is een onderwijsopdracht. Het is **geen beleggingsadvies** en er wordt **niet echt gehandeld**.

## Wat je vandaag oefent (digitale geletterdheid)
**C – Data & dataverwerking**
- redeneren dat data een beperkt beeld van de werkelijkheid geeft (wat zie je wel/niet?)
- onderzoek uitvoeren met een dataset (API-data) om een vraag te beantwoorden
- reflecteren op datagedreven werken (en beperkingen zoals ontbrekende data, vertraging, ruis)

**B – Programmeren & computational thinking**
- taak/doel van een programma beschrijven
- algoritme schetsen (stappenplan van “symbol → data → analyse → plan”)
- datastructuren gebruiken (dict/JSON → DataFrame)
- testen en bijstellen (error handling, andere symbolen/perioden)

---

## Context: het voorbeeld-dashboard
Het voorbeeld-dashboard toont o.a.:
- koersgrafiek over een periode (30D/100D/1Y/2Y)
- **Open/High/Low/Close**
- **Gemiddelde prijs**, **Hoogste**, **Laagste**, **Volatiliteit**
- knoppen: “Quote”, “Add to watchlist”, “Buy/Sell” + “Mijn plan”

**Jullie opdracht (economie):** reverse engineer wat je nodig hebt uit de API om dit te kunnen nabouwen en te kunnen uitleggen.


## 1) Setup – API key veilig gebruiken

We gebruiken een environment variable, zodat de key **niet** in je notebook staat.

Als jouw key al in je systeem staat als `VITE_TWELVE_DATA_KEY`, dan wordt die automatisch gelezen.


In [ ]:
import os
import json
import requests
import pandas as pd
from datetime import datetime

API_KEY = os.getenv("VITE_TWELVE_DATA_KEY") or os.getenv("TWELVEDATA_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "Geen API key gevonden. Zet een env var:
"
        " - VITE_TWELVE_DATA_KEY
"
        "of
"
        " - TWELVEDATA_API_KEY
"
        "Tip (lokaal): export VITE_TWELVE_DATA_KEY='...'
"
    )

BASE_URL = "https://api.twelvedata.com"
print("API key gevonden ✅")

## 2) De kernvraag (computational thinking)
Schrijf (in woorden) een stappenplan/algoritme voor dit programma:

1. Gebruiker kiest een **symbool** (bijv. NFLX).
2. Programma haalt **time series data** op via een API-call.
3. Programma onderzoekt de structuur van de JSON (welke velden bestaan er?).
4. Programma zet data om naar een tabel (DataFrame).
5. Programma haalt de kerninformatie eruit voor het dashboard.
6. Programma helpt de gebruiker een **fictief koop/verkoop-plan** te formuleren.

> Vul hieronder je eigen stappenplan in (in de markdown-cel).  
> Later ga je dit stappenplan vergelijken met de code die je hebt.


**Mijn stappenplan (algoritme):**
- ...


## 3) API-call: time series ophalen

We starten met **1 dag interval**. In de echte wereld bestaan ook andere intervallen (1min, 5min, 1week, …).
Voor reverse engineering is 1 day meestal het duidelijkst.

### Opdracht
- Probeer **minstens 3 verschillende symbolen** (bijv. `AAPL`, `MSFT`, `TSLA`, `NFLX`, `ASML`, …).
- Noteer: wat gebeurt er als je een fout symbool invult?


In [ ]:
def twelvedata_time_series(symbol: str, interval: str = "1day", outputsize: int = 120):
    """Haal time series data op (OHLC) en geef de ruwe JSON terug."""
    params = {
        "symbol": symbol,
        "interval": interval,
        "outputsize": outputsize,
        "apikey": API_KEY
    }
    r = requests.get(f"{BASE_URL}/time_series", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

symbol = "NFLX"  # <-- verander dit
raw = twelvedata_time_series(symbol, interval="1day", outputsize=120)

# Toon de top van de JSON structuur (eerste laag)
raw.keys(), raw.get("status"), raw.get("symbol")

## 4) Reverse engineering: hoe ziet de JSON eruit?

### Opdracht
1. Bekijk de JSON in “mooie” vorm (pretty print).
2. Zoek antwoorden op:
   - Waar staat de **metadata** (symbol, interval, currency, exchange, …)?
   - Waar staan de **waarden per dag**?
   - Welke velden zitten in één dagrecord (open/high/low/close/volume)?
3. Schrijf je bevindingen op (in de markdown-cel).


In [ ]:
# Pretty print (eerste ~1200 tekens om het overzichtelijk te houden)
pretty = json.dumps(raw, indent=2)
print(pretty[:1200])

**Mijn bevindingen over de JSON:**
- Metadata staat bij: ...
- Dagrecords staan bij: ...
- Een dagrecord bevat: ...
- Mogelijke valkuilen / ontbrekende info: ...


## 5) JSON → DataFrame (dataset maken)

We zetten de dagrecords om naar een tabel zodat we makkelijk kunnen analyseren.

### Opdracht
- Controleer of alle kolommen getallen zijn (dus niet tekst).
- Controleer of de datum als echte datum wordt herkend.


In [ ]:
def json_to_df_time_series(raw_json: dict) -> pd.DataFrame:
    values = raw_json.get("values", [])
    if not values:
        raise ValueError(f"Geen 'values' gevonden. Response: {raw_json}")
    df = pd.DataFrame(values)
    # Zet types om
    df["datetime"] = pd.to_datetime(df["datetime"])
    for col in ["open", "high", "low", "close", "volume"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.sort_values("datetime").reset_index(drop=True)
    return df

df = json_to_df_time_series(raw)
df.head(), df.dtypes

## 6) Welke velden heb je nodig voor het dashboard?

Maak een lijstje: “dashboard-element → data die je nodig hebt”.

Voorbeeld:
- “Laagste (30 dagen)” → `min(low)` over de laatste 30 rijen
- “Hoogste (30 dagen)” → `max(high)` over de laatste 30 rijen
- “Gem. prijs (30 dagen)” → gemiddelde van `close` (of OHLC-gemiddelde)

### Opdracht
Vul het schema hieronder in.


**Dashboard mapping**
- Symbool → ...
- Valuta → ...
- Exchange → ...
- Koersgrafiek → ...
- Open/High/Low/Close (laatste dag) → ...
- Gem. prijs (periode) → ...
- Hoogste/Laagste (periode) → ...
- Volatiliteit → (later bij wiskunde notebook; hier mag je alleen beschrijven wat het economisch betekent)


## 7) Dashboard-getallen uitrekenen (beschrijvend)

We rekenen nu alleen **beschrijvende statistiek** uit (hoogste/laagste/gemiddelde en de laatste OHLC).
Volatiliteit rekenen we **nog niet volledig wiskundig** uit; we gebruiken alleen een eenvoudige indicatie.

> Echte volatiliteit is een wiskundig/statistisch begrip. In de wiskunde-notebook ga je onderzoeken hoe je dat precies berekent.


In [ ]:
def dashboard_summary(df: pd.DataFrame, window_days: int = 30) -> dict:
    # Neem de laatste N rijen (ongeveer N handelsdagen, maar niet exact 'kalenderdagen')
    dff = df.tail(window_days).copy()
    latest = df.iloc[-1].to_dict()

    summary = {
        "period_rows": len(dff),
        "avg_close": float(dff["close"].mean()),
        "highest_high": float(dff["high"].max()),
        "lowest_low": float(dff["low"].min()),
        "latest_open": float(latest.get("open")),
        "latest_high": float(latest.get("high")),
        "latest_low": float(latest.get("low")),
        "latest_close": float(latest.get("close")),
        "latest_date": str(latest.get("datetime")),
    }

    # Eenvoudige 'bewegelijkheid'-indicator (niet de officiële volatiliteit):
    # gemiddelde absolute procentuele dagverandering van close
    returns = dff["close"].pct_change()
    summary["avg_abs_daily_move_pct"] = float((returns.abs().mean()) * 100)

    return summary

summary_30 = dashboard_summary(df, window_days=30)
summary_30

## 8) Economische interpretatie (geen rekenen, wel redeneren)

### Opdracht (economie)
Beantwoord in je eigen woorden:
1. Wat betekent **Open/High/Low/Close** economisch? (waarom kunnen ze verschillen?)
2. Wat kun je leren van **hoogste/laagste** in een periode?
3. Waarom geeft een dataset altijd een beperkt beeld van de werkelijkheid? Noem minstens 3 voorbeelden.
   - Denk aan nieuws, emoties op de markt, onverwachte gebeurtenissen, ontbrekende data, nabeurs, dividend, splits, etc.
4. Welke extra info mis je nog om “echt” te kunnen beslissen (zonder dat je die vandaag hoeft te hebben)?

Schrijf je antwoorden hieronder.


**Antwoorden (economie):**
1. ...
2. ...
3. ...
4. ...


## 9) Van data naar een (fictief) koop/verkoop-plan

In het voorbeeld-dashboard staat:
- **Buy below (optional)**: “ik koop als de prijs onder X komt”
- **Sell above (optional)**: “ik verkoop als de prijs boven Y komt”

### Belangrijk
Een plan maken = **regel(s) kiezen** die je kunt uitleggen met data + economische argumenten.
Niet: “ik gok dat het stijgt”.

### Opdracht
Kies één van de onderstaande strategieën (of bedenk zelf een regel):
- **A: Range-regel**: koop dicht bij de laagste koers van de laatste 30 dagen; verkoop dichter bij de hoogste.
- **B: Gemiddelde-regel**: koop als de koers duidelijk onder het 30D-gemiddelde ligt; verkoop als hij erboven ligt.
- **C: Nieuws/reden-regel (qualitatief)**: combineer data met een korte nieuwscheck (handmatig) en leg uit hoe je nieuws je plan beïnvloedt.

Schrijf je regels op + waarom je dat logisch vindt.


**Mijn plan (regels + motivatie):**
- Koop als: ...
- Verkoop als: ...
- Waarom past dit bij dit aandeel / deze periode?: ...
- Risico’s / wanneer werkt dit NIET?: ...


## 10) Maak het plan concreet met drempels (X en Y)

We gebruiken de 30D-samenvatting om een voorstel te doen voor X en Y.
Jij mag die drempels aanpassen, maar je moet het **kunnen uitleggen**.

> Tip: Een simpele start is:
- Buy below = laagste_30d + 25% van (hoogste_30d - laagste_30d)
- Sell above = laagste_30d + 75% van (hoogste_30d - laagste_30d)

Dit is geen 'beste' methode — alleen een **transparante** methode die je kunt onderbouwen.


In [ ]:
low_ = summary_30["lowest_low"]
high_ = summary_30["highest_high"]
range_ = high_ - low_

buy_below = low_ + 0.25 * range_
sell_above = low_ + 0.75 * range_

proposal = {
    "symbol": raw.get("symbol"),
    "period_rows": summary_30["period_rows"],
    "lowest_low_30": low_,
    "highest_high_30": high_,
    "avg_close_30": summary_30["avg_close"],
    "avg_abs_daily_move_pct_30": summary_30["avg_abs_daily_move_pct"],
    "buy_below_proposal": buy_below,
    "sell_above_proposal": sell_above,
    "latest_close": summary_30["latest_close"],
}
proposal

## 11) Test je plan: zou je drempel ooit geraakt zijn?

We controleren in de afgelopen 30 “handelsdagen”:
- hoeveel dagen lag `low` onder buy_below?
- hoeveel dagen lag `high` boven sell_above?

Dit is geen backtest voor winst, maar een sanity check: is je plan überhaupt “realistisch” binnen deze periode?


In [ ]:
buy = proposal["buy_below_proposal"]
sell = proposal["sell_above_proposal"]

dff = df.tail(30).copy()
buy_hits = (dff["low"] <= buy).sum()
sell_hits = (dff["high"] >= sell).sum()

{
    "buy_below": buy,
    "sell_above": sell,
    "buy_hits_last_30_rows": int(buy_hits),
    "sell_hits_last_30_rows": int(sell_hits),
}

## 12) Reflectie: computational thinking & databeperkingen

### Opdracht
1. Welke aannames maak je in je plan? (bijv. “de toekomst lijkt op de afgelopen 30 dagen”)
2. Welke datafouten/valkuilen kan je code hebben? (denk aan missing values, rate limits, verkeerde types, timezone)
3. Welke verbeteringen zou je willen (maar hoeven vandaag niet)?

Schrijf hieronder.


**Reflectie:**
1. ...
2. ...
3. ...


## Bonus: Quote endpoint (voor actuele prijs)

Veel dashboards hebben ook een snelle “Quote” (huidige prijs + dagverandering).
Probeer hieronder de quote-call en vergelijk met je time series (laatste close).

> Let op: actuele prijs kan anders zijn dan laatste close door tijdstip, after-hours, etc.


In [ ]:
def twelvedata_quote(symbol: str) -> dict:
    params = {"symbol": symbol, "apikey": API_KEY}
    r = requests.get(f"{BASE_URL}/quote", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

quote = twelvedata_quote(symbol)
# Kijk welke velden je krijgt:
list(quote.keys())[:25], quote.get("symbol"), quote.get("close"), quote.get("percent_change")

## Afsluiting
- Je hebt de **structuur van stockdata** ontdekt via een API.
- Je hebt data omgezet naar een dataset en dashboard-getallen afgeleid.
- Je hebt een **fictief koop/verkoop-plan** gemaakt met argumenten.

Volgende stap (wiskunde notebook):
- hoe bereken je volatiliteit goed?
- welke berekeningen zitten er “verstopt” achter gemiddelden, moving averages, returns, risico, enz.
